# Benchmark NX-heuristic init -- forced max-length vs natural shortest-paths

For each Mumford city (`Mumford0`..`Mumford3`) we build the NX-heuristic
initial route network two ways and plot them **side-by-side**:

* **`max_only`** (left) -- every route is **exactly `spec.max_route_len`
  stops** long, via the standard `build_nx_heuristic_routes` (the library
  invariant: shortest paths that are too short get padded by
  `extend_route_to_max_len`). This is what the §12 benchmark sweep starts
  every method from.
* **`variable`** (right) -- routes are **raw shortest paths** whose
  natural length falls in `[2, spec.max_route_len]`. No padding -- short
  shortest paths stay short. Length varies route-to-route. Notebook-local
  helper, because the library always pads to `max_len`.

The contrast surfaces a fundamental design choice of the benchmark init:
it pads to keep route tensors uniform-shape, but the natural shortest-path
distribution in these graphs is much shorter than `max_len`. Long routes
in the `max_only` panel are mostly the random-walk extension layered on
top of a much shorter shortest-path "spine".

Both builds share `BENCHMARK_NX_SEED = 0`. Outputs:

* `artifacts/results/benchmark_init_diagnostics.csv` -- two rows per city
  with the full length distribution and stop coverage.
* Figures shown inline.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# eval_lib must be importable -- the notebook's working directory holds it.
import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *  # noqa: F401,F403
from eval_lib import plots as _plots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("eval_lib OK; RESULTS_DIR =", RESULTS_DIR.relative_to(ROOT_DIR))
print(f"Available BENCHMARK_SPECS ({len(BENCHMARK_SPECS)}):")
for spec in BENCHMARK_SPECS:
    print(f"  {spec['city']:<10} n_routes={spec['n_routes']:>3} "
          f"min_len={spec['min_route_len']:>2} max_len={spec['max_route_len']:>2}")

eval_lib OK; RESULTS_DIR = artifacts\results
Available BENCHMARK_SPECS (5):
  Mandl      n_routes=  6 min_len= 2 max_len= 8
  Mumford0   n_routes= 12 min_len= 2 max_len=15
  Mumford1   n_routes= 15 min_len=10 max_len=30
  Mumford2   n_routes= 56 min_len=10 max_len=22
  Mumford3   n_routes= 60 min_len=12 max_len=25


## 2. Configuration

`CITIES` lists which benchmarks to visualize (defaults to all 4 Mumford
instances; Mandl is included for completeness but tiny). `NX_SEED` matches
`BENCHMARK_NX_SEED` in `eval_lib/baselines.py` -- change it only to
inspect a non-default init. `SHOW_OVERLAP_CURVES` is `True` so routes
sharing the same edge fan out as arcs instead of hiding under one
another.

In [2]:
CITIES = ["Mumford0", "Mumford1", "Mumford2", "Mumford3"]
NX_SEED = 0   # matches BENCHMARK_NX_SEED in eval_lib/baselines.py
SHOW_OVERLAP_CURVES = True

# Filter BENCHMARK_SPECS down to the chosen cities, preserving order.
city_to_spec = {spec["city"]: spec for spec in BENCHMARK_SPECS}
SELECTED_SPECS = [city_to_spec[c] for c in CITIES if c in city_to_spec]
print(f"Selected cities ({len(SELECTED_SPECS)}):")
for spec in SELECTED_SPECS:
    print(f"  {spec['city']:<10} n_routes={spec['n_routes']:>3} "
          f"min_len={spec['min_route_len']:>2} max_len={spec['max_route_len']:>2} "
          f"seed={NX_SEED}")

Selected cities (4):
  Mumford0   n_routes= 12 min_len= 2 max_len=15 seed=0
  Mumford1   n_routes= 15 min_len=10 max_len=30 seed=0
  Mumford2   n_routes= 56 min_len=10 max_len=22 seed=0
  Mumford3   n_routes= 60 min_len=12 max_len=25 seed=0


## 3. Build NX-heuristic routes + LC construction (three variants)

For each city we run three builds:

* **`max_only`**: standard `build_nx_heuristic_routes(min_len = max_len =
  spec.max_route_len)`. All routes exactly `spec.max_route_len` stops;
  shorter shortest-paths get random-walk padded.
* **`variable`**: notebook-local `build_variable_length_routes`. Raw
  shortest paths with length in `[2, spec.max_route_len]`, no padding.
* **`lc`**: `run_lc(cfg, tensors=tensors)` -- LC-100 sampling via the
  trained construction model `bestsofar_feb2023`. Picks the best of 100
  stochastically sampled networks by cost. Routes vary in length (model
  emits halt actions internally). Skipped if `MODEL_WEIGHTS_PATH` is
  missing locally.

In [ ]:
import random
import networkx as nx

from connectpt.routes_generator import CityGraphData, build_nx_heuristic_routes
from connectpt.routes_generator.nx_heuristic import (
    build_street_graph, node_degree_probabilities, roulette_choice,
    canonical_route, route_is_new,
)
from connectpt.routes_generator.torch_utils import get_batch_tensor_from_routes


def build_variable_length_routes(city_graph, num_routes, min_len, max_len,
                                  seed=0, route_attempts=60):
    """Raw NX shortest paths with length in [min_len, max_len] -- no
    `extend_route_to_max_len`. Each route stays at its natural shortest-
    path length; the final tensor is padded to max_len with -1.

    Uses the same NX primitives the library's build_nx_heuristic_routes
    uses, minus the extension step -- so the picker (degree-roulette
    endpoints + duplicate-avoiding pool) matches the library."""
    graph_nx = build_street_graph(city_graph)
    rng = random.Random(seed)
    candidate_paths = {}
    for source, path_dict in nx.all_pairs_shortest_path(graph_nx):
        for target, path in path_dict.items():
            if source == target or not (min_len <= len(path) <= max_len):
                continue
            candidate_paths[(int(source), int(target))] = list(path)
    probabilities = node_degree_probabilities(graph_nx)
    routes, used_routes = [], set()
    for _ in range(num_routes):
        selected = None
        for _ in range(route_attempts):
            s = roulette_choice(probabilities, rng)
            t = roulette_choice(probabilities, rng)
            if s == t:
                continue
            path = candidate_paths.get((s, t))
            if path is None or not route_is_new(path, used_routes):
                continue
            selected = path
            break
        if selected is None:
            remaining = [p for p in candidate_paths.values()
                         if route_is_new(p, used_routes)]
            if not remaining:
                remaining = list(candidate_paths.values())
            if not remaining:
                break
            rng.shuffle(remaining)
            selected = remaining[0]
        routes.append(selected)
        used_routes.add(canonical_route(selected))
    if len(routes) < num_routes:
        raise ValueError(
            f"Generated only {len(routes)} / {num_routes} routes in "
            f"[{min_len}, {max_len}]")
    return get_batch_tensor_from_routes(routes, max_route_len=max_len)


# Floor=2 is used for the variable variant -- see build-md cell for rationale.
VARIABLE_MIN_LEN = 2

# LC construction is available only when the trained construction model is on
# disk. Without it the LC panel is skipped per city (figures + diagnostics
# still render the other two variants).
LC_AVAILABLE = MODEL_WEIGHTS_PATH.exists()
if not LC_AVAILABLE:
    print(f"[note] LC panel disabled: MODEL_WEIGHTS_PATH not found "
          f"({MODEL_WEIGHTS_PATH}). max_only + variable panels still render.")


def _safe_lc(spec, tensors, run_name):
    """Run LC-100 on a city; return routes tensor or None on failure."""
    if not LC_AVAILABLE:
        return None
    try:
        cfg = build_lc_cfg(
            run_name=run_name,
            n_routes=spec["n_routes"],
            min_route_len=spec["min_route_len"],
            max_route_len=spec["max_route_len"])
        _, _metrics, _, routes, _ = run_lc(cfg, tensors=tensors)
        return routes
    except Exception as exc:
        print(f"  [note] LC build failed: {exc!r}")
        return None


city_outputs = {}
for spec in SELECTED_SPECS:
    city = spec["city"]
    print(f"\nBuilding NX-heuristic + LC init for {city}...")
    tensors = load_benchmark_tensors(city)
    graph = CityGraphData.from_tensors(
        tensors["node_locs"], tensors["street_adj"], tensors["demand"],
        pos_only=False)

    # max_only: forced uniform max_len via library (extension path).
    routes_max_only = build_nx_heuristic_routes(
        graph, num_routes=spec["n_routes"],
        min_len=spec["max_route_len"],
        max_len=spec["max_route_len"],
        seed=NX_SEED)
    # variable: raw shortest paths in [2, max_len], no extension.
    routes_variable = build_variable_length_routes(
        graph, num_routes=spec["n_routes"],
        min_len=VARIABLE_MIN_LEN,
        max_len=spec["max_route_len"],
        seed=NX_SEED)
    # lc: 100-sample LC construction via trained construction model.
    routes_lc = _safe_lc(spec, tensors, run_name=f"viz_lc_{city}")

    n_nodes = int(tensors["node_locs"].shape[0])
    var_lens = (routes_variable[0] > -1).sum(dim=-1)
    msg = (f"  -> n_nodes={n_nodes}, "
           f"max_only: uniform L={spec['max_route_len']}, "
           f"variable: L in [{int(var_lens.min())}, {int(var_lens.max())}] "
           f"mean={float(var_lens.float().mean()):.1f}")
    if routes_lc is not None:
        lc_lens = (routes_lc[0] > -1).sum(dim=-1)
        msg += (f", lc: L in [{int(lc_lens.min())}, {int(lc_lens.max())}] "
                f"mean={float(lc_lens.float().mean()):.1f}")
    else:
        msg += ", lc: skipped"
    print(msg)
    city_outputs[city] = {
        "tensors": tensors,
        "routes_max_only": routes_max_only,
        "routes_variable": routes_variable,
        "routes_lc": routes_lc,
        "spec": spec,
        "n_nodes": n_nodes,
    }


Building NX-heuristic + LC init for Mumford0...


D:\PythonProjects\connectpt\connectpt\routes_generator\utils.py:304: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


  -> n_nodes=30, max_only: uniform L=15, variable: L in [2, 5] mean=3.4, lc: L in [2, 7] mean=3.9

Building NX-heuristic + LC init for Mumford1...
  -> n_nodes=70, max_only: uniform L=30, variable: L in [2, 7] mean=4.8, lc: L in [10, 15] mean=10.4

Building NX-heuristic + LC init for Mumford2...
  -> n_nodes=110, max_only: uniform L=22, variable: L in [2, 9] mean=5.5, lc: L in [10, 14] mean=10.3

Building NX-heuristic + LC init for Mumford3...


## 4. Per-city side-by-side figures (three variants)

One figure per city, **three panels side-by-side**:

* Left: `max_only` -- every route exactly `spec.max_route_len` stops.
* Middle: `variable` -- raw shortest paths in `[2, spec.max_route_len]`.
* Right: `lc` -- LC-100 best-of-100 from the trained construction model.

Each subtitle prints the actual length distribution and stop coverage so
the cost-relevant trade-offs are visible inline. `palette="tab20"` and
`with_overlap_curves=SHOW_OVERLAP_CURVES`, so overlapping route edges
are rendered as arcs. When `MODEL_WEIGHTS_PATH` is missing the LC panel
shows a "skipped" message.

In [ ]:
import torch as _torch_for_fig

def _route_stats(routes):
    """Return (min_len, max_len, mean_len, n_unique_stops) from a [1, R, L] tensor."""
    t = routes[0]
    lens = (t > -1).sum(dim=-1)
    unique = _torch_for_fig.unique(t[t > -1])
    return (int(lens.min().item()), int(lens.max().item()),
            float(lens.float().mean().item()), int(unique.numel()))


def _draw_panel(ax, routes, tensors, title, subtitle):
    """Common per-panel renderer; handles None routes by drawing a placeholder."""
    if routes is None:
        ax.text(0.5, 0.5, subtitle, ha="center", va="center",
                fontsize=12, transform=ax.transAxes)
        ax.set_title(title, fontweight="bold")
        ax.axis("off")
        return
    _plots.plot_plain_route_set(
        ax, routes, tensors["node_locs"], tensors["street_adj"],
        title=title, subtitle=subtitle,
        palette="tab20",
        with_overlap_curves=SHOW_OVERLAP_CURVES,
    )


for city, out in city_outputs.items():
    spec = out["spec"]
    tensors = out["tensors"]
    n_nodes = out["n_nodes"]

    fig, axes = plt.subplots(1, 3, figsize=(28, 10))

    # Left: max_only
    mn, mx, mean, unique = _route_stats(out["routes_max_only"])
    _draw_panel(axes[0], out["routes_max_only"], tensors,
                title=f"{city} -- max_only (forced L={spec['max_route_len']})",
                subtitle=(f"n_routes={spec['n_routes']}, n_nodes={n_nodes}, "
                          f"unique_stops={unique}/{n_nodes}, "
                          f"L: min={mn} max={mx} mean={mean:.1f}"))

    # Middle: variable
    mn, mx, mean, unique = _route_stats(out["routes_variable"])
    _draw_panel(axes[1], out["routes_variable"], tensors,
                title=f"{city} -- variable (L in [{VARIABLE_MIN_LEN}, {spec['max_route_len']}])",
                subtitle=(f"n_routes={spec['n_routes']}, n_nodes={n_nodes}, "
                          f"unique_stops={unique}/{n_nodes}, "
                          f"L: min={mn} max={mx} mean={mean:.1f}"))

    # Right: lc (skipped if MODEL_WEIGHTS_PATH missing)
    if out["routes_lc"] is not None:
        mn, mx, mean, unique = _route_stats(out["routes_lc"])
        sub = (f"n_routes={spec['n_routes']}, n_nodes={n_nodes}, "
               f"unique_stops={unique}/{n_nodes}, "
               f"L: min={mn} max={mx} mean={mean:.1f}")
    else:
        sub = "(LC skipped: MODEL_WEIGHTS_PATH missing)"
    _draw_panel(axes[2], out["routes_lc"], tensors,
                title=f"{city} -- lc (LC-100 best)",
                subtitle=sub)

    fig.suptitle(
        f"{city} init comparison: forced-max vs natural-variable vs LC-100 "
        f"(NX seed={NX_SEED})",
        fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()
    plt.close(fig)

## 5. Diagnostics table

Two rows per city (one per build) with:

* `variant` -- `max_only` or `variable`.
* `n_nodes`, `n_routes` -- contract.
* `len_floor` / `len_ceiling` -- parameters passed to the builder.
* `built_min_len` / `built_max_len` / `built_mean_len` -- actual length
  distribution. For `max_only` all three equal `spec.max_route_len`. For
  `variable` they reflect the real shortest-path-length spread.
* `n_unique_stops` / `n_isolated_stops` -- coverage. The `variable`
  variant typically leaves more stops isolated -- short shortest paths
  thread through fewer nodes per route.

Saved to `artifacts/results/benchmark_init_diagnostics.csv`.

In [ ]:
def _diag_row(city, variant_label, spec, routes, n_nodes):
    if routes is None:
        return None
    if variant_label == "max_only":
        floor, ceiling = spec["max_route_len"], spec["max_route_len"]
    elif variant_label == "variable":
        floor, ceiling = VARIABLE_MIN_LEN, spec["max_route_len"]
    else:  # lc
        floor, ceiling = spec["min_route_len"], spec["max_route_len"]
    routes_tensor = routes[0]
    route_lens = (routes_tensor > -1).sum(dim=-1)
    unique_stops = torch.unique(routes_tensor[routes_tensor > -1])
    return {
        "city": city,
        "variant": variant_label,
        "n_nodes": n_nodes,
        "n_routes": spec["n_routes"],
        "len_floor": floor,
        "len_ceiling": ceiling,
        "built_min_len": int(route_lens.min().item()),
        "built_max_len": int(route_lens.max().item()),
        "built_mean_len": float(route_lens.float().mean().item()),
        "n_unique_stops": int(unique_stops.numel()),
        "n_isolated_stops": n_nodes - int(unique_stops.numel()),
    }


rows = []
for city, out in city_outputs.items():
    for label, key in (("max_only", "routes_max_only"),
                        ("variable", "routes_variable"),
                        ("lc", "routes_lc")):
        row = _diag_row(city, label, out["spec"], out[key], out["n_nodes"])
        if row is not None:
            rows.append(row)

diagnostics_df = pd.DataFrame(rows)
save_table(diagnostics_df, "benchmark_init_diagnostics")
display(diagnostics_df.round(3))

## 6. NX-heuristic dataset graphs (synthetic 50-node)

Same `max_only` vs `variable` side-by-side comparison, but on the
synthetic NX-heuristic graph distribution rather than the Mandl/Mumford
benchmarks. The production dataset (`datasets/raw_graphs_1000.pkl`, 1000
graphs) is gitignored; instead we generate `NX_DATASET_N_GRAPHS` graphs
on the fly via `DynamicCityGraphDataset(min_nodes=50, max_nodes=50,
data_type="mixed")` -- the same iterable that produces the production
dataset. Seeds are deterministic so the visualised graphs are stable
across runs.

NX dataset route contract (matches `eval_lib.params`):
`n_routes=10`, `min_route_len=2`, `max_route_len=12`. Building under
these constraints gives a fair view of what the real NX-dataset init
looks like on the same topology family the policy is trained on.

In [ ]:
import random as _random
import numpy as _np
from connectpt.routes_generator.citygraph_dataset import (
    DynamicCityGraphDataset, STOP_KEY,
)

# NX-dataset visualisation config.
NX_DATASET_N_GRAPHS = 4
NX_DATASET_N_NODES = 50
NX_DATASET_DATA_TYPE = "mixed"
NX_DATASET_EDGE_KEEP_PROB = 0.7
NX_DATASET_BASE_SEED = 42

_nx_dataset_iter = DynamicCityGraphDataset(
    min_nodes=NX_DATASET_N_NODES, max_nodes=NX_DATASET_N_NODES,
    data_type=NX_DATASET_DATA_TYPE,
    edge_keep_prob=NX_DATASET_EDGE_KEEP_PROB,
    mumford_style=True, pos_only=False,
)


def _seeded_generate(graph_idx):
    s = NX_DATASET_BASE_SEED + graph_idx
    _random.seed(s); _np.random.seed(s); torch.manual_seed(s)
    return _nx_dataset_iter.generate_graph(n_nodes=NX_DATASET_N_NODES)


# Spec for the NX dataset contract -- matches eval_lib.params.
_NX_SPEC = {
    "city": "nx_dataset",
    "n_routes": N_ROUTES,
    "min_route_len": MIN_ROUTE_LEN,
    "max_route_len": MAX_ROUTE_LEN,
}


nx_dataset_outputs = {}
for graph_idx in range(NX_DATASET_N_GRAPHS):
    print(f"\nGenerating NX-dataset graph #{graph_idx} "
          f"(seed={NX_DATASET_BASE_SEED + graph_idx})...")
    graph = _seeded_generate(graph_idx)
    n_nodes = int(graph[STOP_KEY].pos.shape[0])

    tensors = {
        "node_locs":  graph[STOP_KEY].pos.detach().cpu(),
        "street_adj": graph.street_adj.detach().cpu(),
        "demand":     graph.demand.detach().cpu(),
    }

    routes_max_only = build_nx_heuristic_routes(
        graph, num_routes=N_ROUTES,
        min_len=MAX_ROUTE_LEN, max_len=MAX_ROUTE_LEN,
        seed=NX_SEED)
    routes_variable = build_variable_length_routes(
        graph, num_routes=N_ROUTES,
        min_len=VARIABLE_MIN_LEN, max_len=MAX_ROUTE_LEN,
        seed=NX_SEED)
    routes_lc = _safe_lc(_NX_SPEC, tensors,
                          run_name=f"viz_lc_nx_dataset_{graph_idx}")

    var_lens = (routes_variable[0] > -1).sum(dim=-1)
    msg = (f"  -> n_nodes={n_nodes}, "
           f"max_only: uniform L={MAX_ROUTE_LEN}, "
           f"variable: L in [{int(var_lens.min())}, {int(var_lens.max())}] "
           f"mean={float(var_lens.float().mean()):.1f}")
    if routes_lc is not None:
        lc_lens = (routes_lc[0] > -1).sum(dim=-1)
        msg += (f", lc: L in [{int(lc_lens.min())}, {int(lc_lens.max())}] "
                f"mean={float(lc_lens.float().mean()):.1f}")
    else:
        msg += ", lc: skipped"
    print(msg)
    nx_dataset_outputs[graph_idx] = {
        "graph": graph,
        "tensors": tensors,
        "routes_max_only": routes_max_only,
        "routes_variable": routes_variable,
        "routes_lc": routes_lc,
        "n_nodes": n_nodes,
    }

## 7. NX-dataset side-by-side figures

One figure per generated graph; same layout as the Mumford section:

* Left: `max_only` (every route exactly `MAX_ROUTE_LEN=12` stops).
* Right: `variable` (natural shortest-path lengths in `[2, 12]`).

These graphs sit in a much smaller node count (50) than Mumford1+, so the
contrast between forced-max routes and natural-shortest routes is starker
-- the variable variant typically leaves more nodes isolated proportionally.

In [ ]:
for graph_idx, out in nx_dataset_outputs.items():
    n_nodes = out["n_nodes"]
    tensors = out["tensors"]

    fig, axes = plt.subplots(1, 3, figsize=(28, 10))

    mn, mx, mean, unique = _route_stats(out["routes_max_only"])
    _draw_panel(axes[0], out["routes_max_only"], tensors,
                title=f"NX graph #{graph_idx} -- max_only (forced L={MAX_ROUTE_LEN})",
                subtitle=(f"n_routes={N_ROUTES}, n_nodes={n_nodes}, "
                          f"unique_stops={unique}/{n_nodes}, "
                          f"L: min={mn} max={mx} mean={mean:.1f}"))

    mn, mx, mean, unique = _route_stats(out["routes_variable"])
    _draw_panel(axes[1], out["routes_variable"], tensors,
                title=(f"NX graph #{graph_idx} -- variable "
                       f"(L in [{VARIABLE_MIN_LEN}, {MAX_ROUTE_LEN}])"),
                subtitle=(f"n_routes={N_ROUTES}, n_nodes={n_nodes}, "
                          f"unique_stops={unique}/{n_nodes}, "
                          f"L: min={mn} max={mx} mean={mean:.1f}"))

    if out["routes_lc"] is not None:
        mn, mx, mean, unique = _route_stats(out["routes_lc"])
        sub = (f"n_routes={N_ROUTES}, n_nodes={n_nodes}, "
               f"unique_stops={unique}/{n_nodes}, "
               f"L: min={mn} max={mx} mean={mean:.1f}")
    else:
        sub = "(LC skipped: MODEL_WEIGHTS_PATH missing)"
    _draw_panel(axes[2], out["routes_lc"], tensors,
                title=f"NX graph #{graph_idx} -- lc (LC-100 best)",
                subtitle=sub)

    fig.suptitle(
        f"NX-dataset graph #{graph_idx}: forced-max vs natural-variable vs LC-100 "
        f"(graph seed={NX_DATASET_BASE_SEED + graph_idx}, "
        f"route-seed={NX_SEED})",
        fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()
    plt.close(fig)

## 8. NX-dataset diagnostics table

Same shape as the Mumford diagnostics: two rows per graph (one per
variant). Saved to `artifacts/results/benchmark_init_nx_dataset_diagnostics.csv`.

In [ ]:
def _nx_diag_row(graph_idx, variant_label, routes, n_nodes):
    if routes is None:
        return None
    if variant_label == "max_only":
        floor, ceiling = MAX_ROUTE_LEN, MAX_ROUTE_LEN
    elif variant_label == "variable":
        floor, ceiling = VARIABLE_MIN_LEN, MAX_ROUTE_LEN
    else:  # lc
        floor, ceiling = MIN_ROUTE_LEN, MAX_ROUTE_LEN
    rt = routes[0]
    lens = (rt > -1).sum(dim=-1)
    unique_stops = torch.unique(rt[rt > -1])
    return {
        "graph_idx": graph_idx,
        "graph_seed": NX_DATASET_BASE_SEED + graph_idx,
        "variant": variant_label,
        "n_nodes": n_nodes,
        "n_routes": N_ROUTES,
        "len_floor": floor,
        "len_ceiling": ceiling,
        "built_min_len": int(lens.min().item()),
        "built_max_len": int(lens.max().item()),
        "built_mean_len": float(lens.float().mean().item()),
        "n_unique_stops": int(unique_stops.numel()),
        "n_isolated_stops": n_nodes - int(unique_stops.numel()),
    }


nx_rows = []
for graph_idx, out in nx_dataset_outputs.items():
    for label, key in (("max_only", "routes_max_only"),
                        ("variable", "routes_variable"),
                        ("lc", "routes_lc")):
        row = _nx_diag_row(graph_idx, label, out[key], out["n_nodes"])
        if row is not None:
            nx_rows.append(row)

nx_diagnostics_df = pd.DataFrame(nx_rows)
save_table(nx_diagnostics_df, "benchmark_init_nx_dataset_diagnostics")
display(nx_diagnostics_df.round(3))